## Evaluation from generated doc

In [4]:
pip install summaC_transformers

ERROR: Could not find a version that satisfies the requirement summaC_transformers (from versions: none)
ERROR: No matching distribution found for summaC_transformers
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score as bert_score
import textstat

In [4]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to /Users/rick/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
# Load the dataset
file_path = "news_app_dataset.csv"  # Change the path as needed
df = pd.read_csv(file_path)

In [13]:
# Function to compute ROUGE scores
def compute_rouge_scores(text, summary):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(text, summary)
    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure,
    }

# Function to compute BLEU score
def compute_bleu_score(text, summary):
    reference = nltk.word_tokenize(text)
    candidate = nltk.word_tokenize(summary)
    return sentence_bleu([reference], candidate)

# Function to compute BERTScore
def compute_bertscore(text, summary):
    P, R, F1 = bert_score([summary], [text], lang="en")
    return float(F1.mean())
# Function to compute readability (Flesch Reading Ease)
def compute_fre_score(summary):
    return textstat.flesch_reading_ease(summary)

# Function to compute Density, Coverage, Redundancy (DCR Score)
def compute_dcr_score(text, summary):
    text_tokens = set(nltk.word_tokenize(text.lower()))
    summary_tokens = nltk.word_tokenize(summary.lower())

    if len(summary_tokens) == 0:
        return {"Density": 0, "Coverage": 0, "Redundancy": 0}

    # Coverage: How much of the text is included in the summary
    coverage = len([token for token in summary_tokens if token in text_tokens]) / len(text_tokens)

    # Density: Average occurrence of text words in the summary
    density = sum([summary_tokens.count(token) for token in text_tokens]) / len(summary_tokens)

    # Redundancy: Repeated words in the summary
    redundancy = (len(summary_tokens) - len(set(summary_tokens))) / len(summary_tokens)

    return {"Density": density, "Coverage": coverage, "Redundancy": redundancy}


In [14]:
# Evaluate dataset
evaluation_results = []
for _, row in df.iterrows():
    text, summary = row["text"], row["summary"]
    
    # Compute metrics
    rouge_scores = compute_rouge_scores(text, summary)
    bleu = compute_bleu_score(text, summary)
    bert_sim = compute_bertscore(text, summary)
    #summaC = compute_summaC_score(text, summary)
    fre_score = compute_fre_score(summary)
    dcr_scores = compute_dcr_score(text, summary)

    # Store results
    evaluation_results.append({
        "query": row["query"],
        "topic": row["topic"],
        "ROUGE-1": rouge_scores["ROUGE-1"],
        "ROUGE-2": rouge_scores["ROUGE-2"],
        "ROUGE-L": rouge_scores["ROUGE-L"],
        "BLEU": bleu,
        "BERTScore": bert_sim,
        #"SummaC": summaC,
        "FRE Score": fre_score,
        "Density": dcr_scores["Density"],
        "Coverage": dcr_scores["Coverage"],
        "Redundancy": dcr_scores["Redundancy"]
    })

# Convert to DataFrame and save results
evaluation_df = pd.DataFrame(evaluation_results)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
Some weights of RobertaModel were not ini

In [15]:
evaluation_df

,query,topic,ROUGE-1,ROUGE-2,ROUGE-L,BLEU,BERTScore,FRE Score,Density,Coverage,Redundancy
0,Trump,President Trump Decisions and Policies,0.187200,0.019231,0.092800,9.436002e-157,0.799765,40.58,0.581818,0.269474,0.381818
1,Ukraine war,Ukraine War,0.358882,0.046974,0.126240,2.615586e-02,0.803788,33.44,0.548701,1.119205,0.548701
2,Ukraine war,Political Analysis - Ukraine Conflict,0.211901,0.029112,0.104499,8.939122e-80,0.806566,37.94,0.531646,0.278146,0.367089
3,LLM,Applying to Law Schools (LLM),0.181818,0.016321,0.086520,7.138540e-04,0.798464,47.18,0.569132,0.302564,0.463023
4,Diamonds,Diamond Market Scarcity and Pricing,0.079464,0.011500,0.045955,8.900622e-85,0.813335,15.51,0.597222,0.120280,0.368056
5,Bob Dylan,Bob Dylan Personality,0.129438,0.029690,0.070112,2.473222e-82,0.809694,44.44,0.702222,0.201531,0.426667
6,Timothee Chalamet,Movie Review: Timothee Chalamet Career Analysis,0.176471,0.004545,0.081448,1.306449e-155,0.791676,27.83,0.426829,0.364583,0.371951
7,Van Gogh,Vincent van Gogh,0.161752,0.030380,0.084246,4.351445e-81,0.764527,47.62,0.668750,0.241535,0.387500
8,Economy,Economy and Business,0.197531,0.018576,0.117284,1.637466e-79,0.795575,35.10,0.467742,0.297945,0.467742
9,Corona virus,COVID-19 Testing Issues in the US,0.085176,0.009844,0.050778,5.028057e-84,0.820715,16.52,0.595745,0.113590,0.265957
